# E4 · R12 against R11, by image source and by event (evaluation only, no training)

Task **E4**, an evaluation-only check like E1 (go given on 19 Oct 2026). Nothing is trained. It loads R11's and R12's saved models and scores both on the same 645 validation images, split by image source and by disaster event.

**What it checks.** R12 is R11's recipe plus 1,619 extra xBD tiles holding 8,021 destroyed buildings. Overall it scored the same as R11: Destroyed IoU 0.295 against 0.297, with TTA. Two explanations were offered, and neither has been tested:
- **The wrong place.**
  - Every extra tile is xBD, where R11 already reaches Destroyed IoU 0.456.
  - The weak images are the 85 non-xBD ones (0.071). They hold 43% of the destroyed pixels (Turkey's earthquake alone has 37%) and got no extra tiles.
  - If this is right, R12 gains on xBD, most in the events that got the most new destroyed buildings (Palu 2,206, Santa Rosa 796, Woolsey 374), and not on the other images.
- **The wrong problem.** The model's main error is calling rubble background (E1). If this is right, R12 doesn't gain on xBD either, even where the new tiles came from.

**How**
- R11 and R12 run on the 645 validation images with the D4 code: full resolution, standard and 8-view TTA. Each must first reproduce its own D4 CSV.
- One confusion matrix per model and mode for all images, per source (xBD and other) and per event. The metrics are those of every D4 run.
- Each event is listed with the extra tiles and destroyed buildings R12 got for it (from D5's and D5b's manifests).

**Reading, fixed before the run** (TTA, Destroyed IoU; 0.013 is the seed noise, R1 against R6):
- R12 beats R11 on xBD by more than 0.013, but not on the other images: **the wrong place**. The data helped where it was added.
- R12 doesn't beat R11 on xBD by more than 0.013: **not the wrong place**. More xBD destroyed examples don't help at this training budget, even where they were added.
- Per event: the rank correlation between the extra destroyed buildings and the change in Destroyed IoU, over the xBD events with at least 10,000 destroyed pixels.

**How to run on Kaggle** (any account with the `HF_TOKEN` secret, since R11 and R12 are in AbrarAlam's private repos)
1. Settings: GPU T4 ×1, internet ON, secret `HF_TOKEN` (AbrarAlam).
2. Input: the DisasterM3 mirror (attached automatically). Optional, if Hugging Face fails: either model's `final_model.pth` as a dataset, in a folder named `R11`/`d4_R11` or `R12`/`d4_R12`.
3. Run All once with `SMOKE_TEST = True` (8 images, a few minutes). Then set it to `False` and use Save & Run All: about 20–30 minutes.
4. Outputs in `/kaggle/working/E4/`: `E4_by_source.csv`, `E4_by_event.csv`, `config.json`, `README.md`.

In [1]:
# ── Cell 1: Environment Setup (Run-All safe) ──────────────────────────────
# segmentation-models-pytorch 0.3.4 is needed to load R11 and R12 (their checkpoint key layout).
import importlib.metadata as _md
import os, sys, subprocess

PINS = {
    "segmentation-models-pytorch": "0.3.4",   # baseline version (checkpoint key layout)
}

def _mismatched():
    out = []
    for pkg, want in PINS.items():
        try:
            have = _md.version(pkg)
        except _md.PackageNotFoundError:
            have = "not installed"
        if have != want:
            out.append(f"{pkg}: {have} → {want}")
    return out

if _mismatched():
    print("⏳ Installing pinned versions:")
    for _m in _mismatched():
        print(f"   {_m}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    *(f"{k}=={v}" for k, v in PINS.items()), "huggingface_hub[hf_xet]"],
                   check=True)                # pip failure stops the run here
    if _mismatched():
        raise RuntimeError(f"Pins still wrong after install: {_mismatched()}")
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") == "Batch":
        # background "Save & Run All": nothing imported yet, so new versions load on first import
        print("✓ Dependencies installed (background run) — continuing without restart.")
    else:
        raise SystemExit("✓ Dependencies installed. RESTART THE KERNEL NOW "
                         "(Run → Restart & clear cell outputs), then Run All again.")
else:
    print("✓ All pinned versions already installed — proceeding.")

import torch
print(f"  torch {torch.__version__}, python {sys.version.split()[0]}")

⏳ Installing pinned versions:
   segmentation-models-pytorch: not installed → 0.3.4
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.5/109.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 28.2 MB/s eta 0:00:00
✓ Dependencies installed (background run) — continuing without restart.
  torch 2.11.0+cu128, python 3.13.15


In [2]:
# ── Cell 2: Configuration and inputs ──────────────────────────────────────
import glob
import json
import os
import re
import time
from pathlib import Path

import numpy as np
import torch
from huggingface_hub import hf_hub_download

DATA_ROOT = Path("/kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct")
MANIFEST_PATH = DATA_ROOT / "train_release.json"
OUT_DIR = Path("/kaggle/working/E4")

NUM_CLASSES = 4                     # 0=Background, 1=Intact, 2=Damaged, 3=Destroyed
CLASS_NAMES = ["Background", "Intact", "Damaged", "Destroyed"]
SMOKE_TEST = False                  # True = 8 validation images
MODELS = {"R11": {"std": (0.5811, 0.2911), "tta": (0.5875, 0.2965)},   # D4_R11.csv: (mIoU, Destroyed IoU)
          "R12": {"std": (0.5820, 0.2898), "tta": (0.5883, 0.2945)}}   # D4_R12.csv
HF_USER = "AbrarAlam"               # checkpoints: {HF_USER}/disasterm3-d4-<run>/final_model.pth
NOISE = 0.013                       # Destroyed IoU moves this much from the seed alone (R1 against R6, Phase2_Plan §1)
MIN_EVENT_DESTROYED_PX = 10_000     # smaller events are too noisy for the per-event comparison
# R12's extra xBD tiles per event: (tiles, destroyed polygons), from D5's and D5b's manifest.json
EXTRA = {"joplin_tornado": (148, 3274), "palu_tsunami": (82, 2206), "santa_rosa_wildfire": (41, 796),
         "woolsey_fire": (60, 374), "hurricane_matthew": (56, 227), "tuscaloosa_tornado": (60, 210),
         "moore_tornado": (60, 208), "lower_puna_volcano": (60, 173), "sunda_tsunami": (44, 161),
         "hurricane_michael": (45, 99), "socal_fire": (467, 81), "portugal_wildfire": (60, 61),
         "nepal_flooding": (60, 48), "pinery_bushfire": (60, 40), "hurricane_harvey": (61, 31),
         "guatemala_volcano": (18, 25), "midwest_flooding": (65, 5), "mexico_earthquake": (121, 2),
         "hurricane_florence": (51, 0)}
assert sum(t for t, _ in EXTRA.values()) == 1619 and sum(d for _, d in EXTRA.values()) == 8021

def hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return os.environ.get("HF_TOKEN")

def find_checkpoint(name):
    """As in E1 and R16: a /kaggle/input folder named R11 / d4_R11 / R11-results, then Hugging Face."""
    wanted = {name.lower(), f"d4_{name}".lower(), f"{name}-results".lower()}
    hits = [p for p in glob.glob("/kaggle/input/**/*.pth", recursive=True)
            if any(part.lower() in wanted for part in Path(p).parts)]
    hits.sort(key=lambda p: Path(p).name != "final_model.pth")   # prefer final_model.pth
    if hits:
        return hits[0]
    repo = f"{HF_USER}/disasterm3-d4-{name}"
    errors = []
    for filename in ("final_model.pth", "last.pth"):   # last.pth holds the same epoch-40 weights
        try:
            return hf_hub_download(repo, filename, token=hf_token())
        except Exception as e:
            errors.append(f"{filename}: {type(e).__name__}")
    raise FileNotFoundError(f"No checkpoint for {name}: nothing under /kaggle/input and {repo} gave {errors}. "
                            f"Add a Kaggle dataset with {name}/final_model.pth.")

CKPTS = {name: find_checkpoint(name) for name in MODELS}
OUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ E4: R12 against R11 by source and event{'  [SMOKE TEST]' if SMOKE_TEST else ''}")
for name, path in CKPTS.items():
    print(f"  {name}: {path}")
print(f"  Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}; outputs: {OUT_DIR}")

final_model.pth: reconstructing file:   0%|          |  0.00B /  130MB            

final_model.pth: downloading bytes:           |  0.00B            

final_model.pth: reconstructing file:   0%|          |  0.00B /  130MB            

final_model.pth: downloading bytes:           |  0.00B            

✓ E4: R12 against R11 by source and event
  R11: /root/.cache/huggingface/hub/models--AbrarAlam--disasterm3-d4-R11/snapshots/31974dacfee2125a66639858433a113704edf9b3/final_model.pth
  R12: /root/.cache/huggingface/hub/models--AbrarAlam--disasterm3-d4-R12/snapshots/6f8f2e3d1b181e595de2b81d32973dc64c4317bd/final_model.pth
  Device: Tesla T4; outputs: /kaggle/working/E4


In [3]:
# ── Cell 3: Load manifest and build the 4-class masks (verbatim from R11's Cell 4) ──
import cv2
import numpy as np
import os
from collections import Counter, defaultdict
from pathlib import Path
import json

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)
print(f"Loaded {len(raw_data):,} entries from manifest")

# ── Filter: ONLY Building Damage Assessment segmentation entries ──
bda_entries = [
    e for e in raw_data
    if e.get("task") == "Referring Expression Segmentation"
    and e.get("cls_description") == "Building Damage Assessment"
]
print(f"Building Damage Assessment entries: {len(bda_entries):,}")

# ── Path resolution (same logic as before) ──
def resolve_path(rel_path):
    if not rel_path:
        return None
    rel_path = rel_path.replace("\\", "/")
    filename = Path(rel_path).name
    candidates = [
        DATA_ROOT / rel_path,
        DATA_ROOT / "train_images" / rel_path,
        DATA_ROOT / "train_images" / "train_images" / filename,
        DATA_ROOT / "masks" / rel_path,
        DATA_ROOT / "masks" / "masks" / filename,
    ]
    for c in candidates:
        if c.exists():
            return str(c)
    return None

# ── Group entries by underlying post-disaster image ──
by_image = defaultdict(dict)  # post_image_path -> {folder_type: mask_path}

for e in bda_entries:
    post_img = e.get("post_image_path", "")
    if e.get("image_type") != "Optical":   # skip SAR duplicates, keep optical only
        continue
    mask_rel = e.get("ground_truth", "")
    folder = Path(mask_rel.replace("\\", "/")).parent.name  # e.g. train_building_intact_mask
    resolved = resolve_path(mask_rel)
    if resolved:
        by_image[post_img][folder] = resolved
        by_image[post_img]["post_image_path"] = post_img
        by_image[post_img]["pre_image_path"] = e.get("pre_image_path")   # R11: the before-image

print(f"Unique images with at least one mask: {len(by_image):,}")

# ── Build combined (image, mask) pairs ──
FOLDER_TO_CLASS = {
    "train_building_intact_mask": 1,
    "train_building_damaged_mask": 2,
    "train_building_destroyed_mask": 3,   # use 2 here instead if you want to collapse into "damaged"
}

# ── FIX: Create temporary directory on Kaggle hard drive for masks ──
os.makedirs("/tmp/combined_masks", exist_ok=True)

pairs = []
skipped_img = 0

for post_img, mask_dict in by_image.items():
    img_path = resolve_path(post_img)
    if not img_path:
        skipped_img += 1
        continue

    combined_mask = None
    for folder, class_id in FOLDER_TO_CLASS.items():
        mask_path = mask_dict.get(folder)
        if mask_path is None:
            continue
        m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if m is None:
            continue
        if combined_mask is None:
            combined_mask = np.zeros_like(m, dtype=np.uint8)
        combined_mask[m > 0] = class_id   # later classes overwrite earlier if they overlap

    if combined_mask is None:
        continue  # no valid masks found for this image at all

    # ── FIX: Save mask to hard drive instead of RAM ──
    mask_filename = Path(img_path).name
    save_path = f"/tmp/combined_masks/{mask_filename}"
    cv2.imwrite(save_path, combined_mask)

    pairs.append({
        "image_path": img_path,
        "mask_path": save_path,   # Store the string path, NOT the array!
        "pre_image_path": resolve_path(mask_dict.get("pre_image_path")),   # R11: None if not found
    })

print(f"\n✓ Built {len(pairs):,} valid (image, mask) pairs")
print(f"  Skipped (image not found): {skipped_img:,}")
print(f"  Before-image not found: {sum(p['pre_image_path'] is None for p in pairs):,} (must be 0 for R11)")

# Sanity check
if pairs:
    print(f"\n  Sample: {pairs[0]['image_path']}")
    print(f"  Before: {pairs[0]['pre_image_path']}")
    print(f"  Mask saved at: {pairs[0]['mask_path']}")

Loaded 92,968 entries from manifest
Building Damage Assessment entries: 14,531
Unique images with at least one mask: 6,443

✓ Built 6,443 valid (image, mask) pairs
  Skipped (image not found): 0
  Before-image not found: 0 (must be 0 for R11)

  Sample: /kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct/train_images/train_images/bata_explosion_post_0.png
  Before: /kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct/train_images/train_images/bata_explosion_pre_0.png
  Mask saved at: /tmp/combined_masks/bata_explosion_post_0.png


In [4]:
# ── Cell 4: Validation split (identical to every D4 run) and event names ──
from sklearn.model_selection import train_test_split

train_pairs, val_pairs = train_test_split(pairs, test_size=0.1, random_state=42)
if SMOKE_TEST:
    val_pairs = val_pairs[:8]
missing_pre = sum(p["pre_image_path"] is None for p in val_pairs)
assert missing_pre == 0, f"{missing_pre} validation pairs have no before-image"

XBD_NAME = re.compile(r"^(.+?)_(\d{8})_post_disaster")
OTHER_NAME = re.compile(r"^(.+?)_post")

def event_and_source(image_path):
    """Same rule as E1: 'hurricane_florence_00000261_post_disaster.png' -> ('hurricane_florence', 'xBD')."""
    name = Path(image_path).name
    m = XBD_NAME.match(name)
    if m:
        return m.group(1), "xBD"
    m = OTHER_NAME.match(name)
    return re.sub(r"\d+$", "", m.group(1) if m else Path(name).stem), "other"

events = [event_and_source(p["image_path"]) for p in val_pairs]
n_xbd = sum(1 for _, source in events if source == "xBD")
print(f"✓ Validation images: {len(val_pairs)} ({n_xbd} xBD, {len(val_pairs) - n_xbd} other) in "
      f"{len(set(e for e, _ in events))} events. Expected 645 (560 xBD, 85 other) when SMOKE_TEST is False.")

✓ Validation images: 645 (560 xBD, 85 other) in 24 events. Expected 645 (560 xBD, 85 other) when SMOKE_TEST is False.


In [5]:
# ── Cell 5: The two U-Nets (the D4 prediction code, as in R16) and the E-protocol metrics ──
import segmentation_models_pytorch as smp
import torch.nn.functional as F

def read_triplet(pair):
    """After-image and before-image (RGB, same size) and the 4-class mask, as in the D4 dataset class."""
    post, pre = cv2.imread(pair["image_path"]), cv2.imread(pair["pre_image_path"])
    gt = cv2.imread(pair["mask_path"], cv2.IMREAD_GRAYSCALE)
    if post is None or pre is None or gt is None:
        return None
    post, pre = cv2.cvtColor(post, cv2.COLOR_BGR2RGB), cv2.cvtColor(pre, cv2.COLOR_BGR2RGB)
    h, w = post.shape[:2]
    if pre.shape != post.shape:
        pre = cv2.resize(pre, (w, h), interpolation=cv2.INTER_LINEAR)
    if gt.shape != (h, w):
        gt = cv2.resize(gt, (w, h), interpolation=cv2.INTER_NEAREST)
    return post, pre, gt

def load_unet(path):
    """D4 checkpoint: U-Net + ResNet-50, 6 input channels (after-image, then before-image), 4 classes."""
    state = torch.load(path, map_location=DEVICE, weights_only=False)
    if isinstance(state, dict) and "model_state_dict" in state:
        state = state["model_state_dict"]
    model = smp.Unet(encoder_name="resnet50", encoder_weights=None, in_channels=6, classes=NUM_CLASSES)
    model.load_state_dict(state)
    return model.to(DEVICE).eval()

IMNET_MEAN = np.array([0.485, 0.456, 0.406], np.float32) * 255
IMNET_STD = np.array([0.229, 0.224, 0.225], np.float32) * 255

def unet_input(post, pre):
    """As R11's val_transform: ImageNet normalisation, channels 0-2 after-image, 3-5 before-image."""
    x = np.concatenate([(post.astype(np.float32) - IMNET_MEAN) / IMNET_STD,
                        (pre.astype(np.float32) - IMNET_MEAN) / IMNET_STD], 2)
    return torch.from_numpy(x.transpose(2, 0, 1).copy()).unsqueeze(0).to(DEVICE)

def _forward_once(model, x):
    """Logits at x's own HxW: pad to a multiple of 32, exactly as in the D4 notebooks."""
    h, w = x.shape[-2:]
    ph, pw = (32 - h % 32) % 32, (32 - w % 32) % 32
    if ph or pw:
        x = F.pad(x, (0, pw, 0, ph), mode="reflect")
    return model(x)[..., :h, :w].float()

# The D4 notebooks' 8 dihedral views: rotation 0/90/180/270 × (no flip, horizontal flip)
TTA_VIEWS = [(k, flip) for flip in (False, True) for k in range(4)]

def _tta_fwd(x, k, flip):
    x = torch.rot90(x, k, dims=(2, 3))
    return torch.flip(x, dims=(3,)) if flip else x

def _tta_inv(y, k, flip):
    if flip:
        y = torch.flip(y, dims=(3,))
    return torch.rot90(y, -k, dims=(2, 3))

@torch.no_grad()
def predict_std_and_tta(model, x):
    """One pass over the 8 views (as in E1). The identity view (k=0, no flip) is the standard prediction."""
    prob_sum, pred_std = 0, None
    with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
        for k, flip in TTA_VIEWS:
            logits = _forward_once(model, _tta_fwd(x, k, flip))
            if k == 0 and not flip:
                pred_std = logits.argmax(1)
            prob_sum = prob_sum + torch.softmax(_tta_inv(logits, k, flip), dim=1)
    pred_tta = (prob_sum / len(TTA_VIEWS)).argmax(1)
    return (pred_std[0].cpu().numpy().astype(np.uint8),
            pred_tta[0].cpu().numpy().astype(np.uint8))

def confusion(gt, pred):
    """4×4 pixel counts, rows = truth, columns = prediction."""
    idx = gt.astype(np.int64).ravel() * NUM_CLASSES + pred.astype(np.int64).ravel()
    return np.bincount(idx, minlength=NUM_CLASSES ** 2).reshape(NUM_CLASSES, NUM_CLASSES)

def metrics(cm):
    """Dataset-level metrics from a confusion matrix (E1's formulas), plus building IoU (classes 1-3 merged)."""
    cm = cm.astype(np.float64)
    eps = 1e-8
    tp = np.diag(cm)
    fp = cm.sum(0) - tp
    fn = cm.sum(1) - tp
    iou = tp / (tp + fp + fn + eps)
    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)
    building_tp = cm[1:, 1:].sum()
    building_iou = building_tp / (building_tp + cm[0, 1:].sum() + cm[1:, 0].sum() + eps)
    return {"miou": float(iou.mean()), "iou": iou, "precision": precision, "recall": recall,
            "building_iou": float(building_iou)}

_toy = np.array([[0, 1], [2, 3]], np.uint8)
assert (metrics(confusion(_toy, _toy))["iou"] > 0.99).all()
unets = {name: load_unet(path) for name, path in CKPTS.items()}
print(f"✓ {' and '.join(unets)} loaded (metric self-check passed)")

✓ R11 and R12 loaded (metric self-check passed)


In [6]:
# ── Cell 6: Both U-Nets on every validation image, standard and 8-view TTA ──
from tqdm.auto import tqdm

VARIANTS = [f"{name}_{mode}" for name in MODELS for mode in ("std", "tta")]
GROUPS = ["all", "xBD", "other", *sorted({event for event, _ in events})]
cms = {v: {g: np.zeros((NUM_CLASSES, NUM_CLASSES), np.int64) for g in GROUPS} for v in VARIANTS}
images_per_event = Counter(event for event, _ in events)
source_of_event = {event: source for event, source in events}

t0 = time.time()
for pair, (event, source) in tqdm(list(zip(val_pairs, events)), desc="Validation"):
    data = read_triplet(pair)
    if data is None:
        continue
    post, pre, gt = data
    x = unet_input(post, pre)
    for name, net in unets.items():
        pred_std, pred_tta = predict_std_and_tta(net, x)
        for mode, pred in (("std", pred_std), ("tta", pred_tta)):
            cm = confusion(gt, pred)
            for g in ("all", source, event):
                cms[f"{name}_{mode}"][g] += cm
VAL_MIN = (time.time() - t0) / 60
print(f"✓ {len(val_pairs)} validation images, {len(unets)} U-Nets, standard and TTA, in {VAL_MIN:.1f} min")

Validation:   0%|          | 0/645 [00:00<?, ?it/s]

✓ 645 validation images, 2 U-Nets, standard and TTA, in 12.1 min


In [7]:
# ── Cell 7: By source and by event, the sanity checks, the reading and the output files ──
import pandas as pd
from scipy.stats import spearmanr

def as_markdown(df):
    try:
        return df.to_markdown(index=False)
    except ImportError:              # `tabulate` missing: fall back to a plain-text table
        return "```\n" + df.to_string(index=False) + "\n```"

def scores(cm):
    m = metrics(cm)
    return {"miou": round(m["miou"], 4), **{f"{c.lower()}_iou": round(float(m["iou"][i]), 4) for i, c in enumerate(CLASS_NAMES)},
            "destroyed_recall": round(float(m["recall"][3]), 4), "true_destroyed_px": int(cm[3].sum())}

# 1) Sanity check: each model must reproduce its own D4 CSV
reproduced = {name: all(abs(metrics(cms[f"{name}_{mode}"]["all"])["miou"] - exp[0]) < 5e-4
                        and abs(metrics(cms[f"{name}_{mode}"]["all"])["iou"][3] - exp[1]) < 5e-4
                        for mode, exp in modes.items())
              for name, modes in MODELS.items()}
sanity = ("n/a (smoke test)" if SMOKE_TEST
          else ", ".join(f"{name} {'yes' if ok else 'NO'}" for name, ok in reproduced.items()))
print(f"Each model reproduces its D4 CSV: {sanity}")
if not SMOKE_TEST and not all(reproduced.values()):
    print("⚠ A model doesn't reproduce its CSV: check its checkpoint before reading anything below.")

# 2) By source, standard and TTA, with R12 minus R11
images = {"all": len(val_pairs), "xBD": n_xbd, "other": len(val_pairs) - n_xbd}
rows = []
for g in ("all", "xBD", "other"):
    for mode in ("std", "tta"):
        a, b = scores(cms[f"R11_{mode}"][g]), scores(cms[f"R12_{mode}"][g])
        rows.append({"group": g, "mode": mode, "images": images[g], "true_destroyed_px": a["true_destroyed_px"],
                     "R11_miou": a["miou"], "R12_miou": b["miou"], "miou_change": round(b["miou"] - a["miou"], 4),
                     "R11_destroyed_iou": a["destroyed_iou"], "R12_destroyed_iou": b["destroyed_iou"],
                     "destroyed_change": round(b["destroyed_iou"] - a["destroyed_iou"], 4),
                     "R11_damaged_iou": a["damaged_iou"], "R12_damaged_iou": b["damaged_iou"],
                     "R11_destroyed_recall": a["destroyed_recall"], "R12_destroyed_recall": b["destroyed_recall"]})
by_source = pd.DataFrame(rows)
print("\nBy source:")
print(by_source.to_string(index=False))

# 3) By event (TTA), with the extra tiles R12 got for that event
event_rows = []
for event in GROUPS[3:]:
    a, b = scores(cms["R11_tta"][event]), scores(cms["R12_tta"][event])
    tiles, destroyed = EXTRA.get(event, (0, 0))
    event_rows.append({"event": event, "source": source_of_event[event], "images": images_per_event[event],
                       "true_destroyed_px": a["true_destroyed_px"], "extra_tiles": tiles, "extra_destroyed": destroyed,
                       "R11_destroyed_iou": a["destroyed_iou"], "R12_destroyed_iou": b["destroyed_iou"],
                       "destroyed_change": round(b["destroyed_iou"] - a["destroyed_iou"], 4),
                       "R11_miou": a["miou"], "R12_miou": b["miou"]})
by_event = pd.DataFrame(event_rows).sort_values("true_destroyed_px", ascending=False)
print("\nBy event (with TTA):")
print(by_event.to_string(index=False))

# 4) The reading, by the rule fixed in the first cell (TTA, Destroyed IoU)
tta = by_source[by_source["mode"] == "tta"].set_index("group")
d_xbd, d_other = float(tta.loc["xBD", "destroyed_change"]), float(tta.loc["other", "destroyed_change"])
if SMOKE_TEST:
    reading = "n/a (smoke test)"
elif d_xbd > NOISE and d_other <= NOISE:
    reading = (f"The wrong place: R12 gains {d_xbd:+.3f} in Destroyed IoU on the xBD images, where its extra tiles "
               f"came from, but {d_other:+.3f} on the other images. The data helped where it was added; the weak "
               "spot needs data like the non-xBD events.")
elif d_xbd > NOISE:
    reading = (f"R12 gains on the xBD images ({d_xbd:+.3f}) and on the other images ({d_other:+.3f}) in "
               "Destroyed IoU, more than the seed noise, although its overall score barely moved: read the table.")
else:
    reading = (f"Not the wrong place: R12 changes Destroyed IoU by {d_xbd:+.3f} on the xBD images, within the seed "
               f"noise ({NOISE}), although every extra tile is xBD ({d_other:+.3f} on the others). More xBD "
               "destroyed examples don't help at this training budget, even where they were added.")
big = by_event[(by_event["source"] == "xBD") & (by_event["true_destroyed_px"] >= MIN_EVENT_DESTROYED_PX)]
rho, p_value = spearmanr(big["extra_destroyed"], big["destroyed_change"]) if len(big) >= 4 else (float("nan"), float("nan"))
correlation = (f"Across the {len(big)} xBD events with at least {MIN_EVENT_DESTROYED_PX:,} destroyed pixels, the rank "
               f"correlation between R12's extra destroyed buildings and its change in Destroyed IoU is {rho:+.2f} "
               f"(p = {p_value:.2f}).")
print(f"\n{reading}\n{correlation}")

# 5) Output files
by_source.to_csv(OUT_DIR / "E4_by_source.csv", index=False)
by_event.to_csv(OUT_DIR / "E4_by_event.csv", index=False)
config = {"task": "E4", "smoke_test": SMOKE_TEST, "val_images": len(val_pairs), "checkpoints": CKPTS,
          "reproduces_csv": sanity, "noise": NOISE, "min_event_destroyed_px": MIN_EVENT_DESTROYED_PX,
          "destroyed_change_tta": {"xBD": d_xbd, "other": d_other}, "reading": reading,
          "rank_correlation": {"rho": None if np.isnan(rho) else round(float(rho), 3),
                               "p": None if np.isnan(p_value) else round(float(p_value), 3), "events": len(big)},
          "validation_minutes": round(VAL_MIN, 1)}
(OUT_DIR / "config.json").write_text(json.dumps(config, indent=1), encoding="utf-8")
(OUT_DIR / "README.md").write_text("\n".join([
    "# E4 results: R12 against R11, by source and by event", "",
    f"{len(val_pairs)} validation images{' (SMOKE TEST)' if SMOKE_TEST else ''}, E-protocol (full resolution, "
    "dataset-level confusion matrix per group). R12 is R11's recipe plus 1,619 extra xBD tiles (8,021 destroyed "
    "buildings).", "",
    f"Each model reproduces its D4 CSV: **{sanity}**.", "",
    "## By source", "", as_markdown(by_source), "",
    f"**Reading** (rule fixed before the run; TTA; seed noise {NOISE}): {reading}", "",
    "## By event (with TTA)", "",
    "`extra_tiles` and `extra_destroyed`: what R12 got for the event (D5 and D5b).", "",
    as_markdown(by_event), "", correlation, "",
    "## Files", "",
    "- `E4_by_source.csv`, `E4_by_event.csv`, `config.json`",
]), encoding="utf-8")
print(f"\n✓ Wrote E4_by_source.csv, E4_by_event.csv, config.json and README.md to {OUT_DIR}")

Each model reproduces its D4 CSV: R11 yes, R12 yes

By source:
group mode  images  true_destroyed_px  R11_miou  R12_miou  miou_change  R11_destroyed_iou  R12_destroyed_iou  destroyed_change  R11_damaged_iou  R12_damaged_iou  R11_destroyed_recall  R12_destroyed_recall
  all  std     645            1808111    0.5811    0.5820       0.0009             0.2911             0.2898           -0.0013           0.3828           0.3894                0.3882                0.3835
  all  tta     645            1808111    0.5875    0.5883       0.0008             0.2965             0.2945           -0.0020           0.3956           0.3996                0.3906                0.3834
  xBD  std     560            1029026    0.6369    0.6446       0.0077             0.4446             0.4551            0.0105           0.4556           0.4702                0.6131                0.6040
  xBD  tta     560            1029026    0.6462    0.6525       0.0063             0.4562             0.4628         